# Plump AI — Colab training

Trains the plump-ai agent (Rust game engine + PyTorch PPO) on a Colab GPU,
so the unstable local GPU is not needed for long runs.

**Setup steps:**
1. `Runtime -> Change runtime type` -> **GPU** (T4 / L4 / A100).
2. Run the cells top to bottom. Cell 4 builds the Rust bridge (~2-3 min, once per session).
3. Training checkpoints are saved to **Google Drive** (`MyDrive/plump-ai/checkpoints`), so dropped sessions never lose progress — just re-run the resume cell.

**Notes:**
- Free Colab sessions disconnect after ~1-2 h idle / ~12 h max; GPU quota is limited per day. Each training run is only a few minutes.
- Repo: https://github.com/ollezetterstrom/plump-ai

**Workflow:** run fresh training (cell 6) → check the eval deltas vs random/heuristic → when it plateaus, either stop or start a longer run from the last checkpoint (cell 7) → download the best weights (cell 9) and run them locally.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%bash
cd /content
if [ ! -d plump-ai ]; then
  git clone https://github.com/ollezetterstrom/plump-ai.git
fi
cd plump-ai
git pull --ff-only || true

Already up to date.


Cloning into 'plump-ai'...


In [3]:
%%bash
# Rust toolchain (to compile the PyO3 bridge) — one-time per session
if ! command -v cargo > /dev/null 2>&1; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
export PATH="$HOME/.cargo/bin:$PATH"

cd /content/plump-ai
# venv that reuses Colab's system torch (CUDA build)
sudo apt-get install -y -q python3-venv
python -m venv --system-site-packages .venv
.venv/bin/pip install -q --upgrade maturin pytest

# libpython symlink needed at link time (harmless if already present)
sudo apt-get install -y -q python3-dev > /dev/null 2>&1 || true

VIRTUAL_ENV="$PWD/.venv" .venv/bin/maturin develop --release --manifest-path crates/plump-py/Cargo.toml


  stable-x86_64-unknown-linux-gnu installed - rustc 1.97.1 (8bab26f4f 2026-07-14)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh


info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to minimal
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-07-16 for version 1.97.1 (8bab26f4f 2026-07-14)
info: downloading 3 components
info: default toolchain set to stable-x86_64-unknown-linux-gnu
Error: Command '['/content/plump-ai/.venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
bash: line 10: .venv/bin/pip: No such file or directory
bash: line 15: .venv/bin/maturin: No such file or directory


CalledProcessError: Command 'b'# Rust toolchain (to compile the PyO3 bridge) \xe2\x80\x94 one-time per session\nif ! command -v cargo > /dev/null 2>&1; then\n  curl --proto \'=https\' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal\nfi\nexport PATH="$HOME/.cargo/bin:$PATH"\n\ncd /content/plump-ai\n# venv that reuses Colab\'s system torch (CUDA build)\npython -m venv --system-site-packages .venv\n.venv/bin/pip install -q --upgrade maturin pytest\n\n# libpython symlink needed at link time (harmless if already present)\nsudo apt-get install -y -q python3-dev > /dev/null 2>&1 || true\n\nVIRTUAL_ENV="$PWD/.venv" .venv/bin/maturin develop --release --manifest-path crates/plump-py/Cargo.toml\n'' returned non-zero exit status 127.

In [ ]:
%%bash
# Smoke test: engine import + full test suite
cd /content/plump-ai
.venv/bin/python -c "import plump._engine as e; r = e.Rollout(4, 5, 8, 0); print('engine OK')"
.venv/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
.venv/bin/python -m pytest -q python/tests 2>&1 | tail -2

## Train

Runs `iters` iterations of per-seat PPO (P=4 players, C=5 cards), evaluating vs random every 10 iters and saving `latest.pt` (every iter) + `best.pt` (on improvement) to Drive. Adjust `--iters` freely; a fresh run below starts from a random net.

In [ ]:
%%bash
cd /content/plump-ai
CKPT_DIR="/content/drive/MyDrive/plump-ai/checkpoints"
mkdir -p "$CKPT_DIR"
.venv/bin/python -m plump.train \
  --iters 100 --batch 4096 --players 4 --cards 5 \
  --eval-games 2048 --eval-every 10 --stall 3 \
  --save-dir "$CKPT_DIR"

In [ ]:
%%bash
# Resume/continue from the last Drive checkpoint (re-run this cell as often as you like)
cd /content/plump-ai
CKPT_DIR="/content/drive/MyDrive/plump-ai/checkpoints"
mkdir -p "$CKPT_DIR"
.venv/bin/python -m plump.train \
  --iters 100 --batch 4096 --players 4 --cards 5 \
  --eval-games 2048 --eval-every 10 --stall 3 \
  --save-dir "$CKPT_DIR" --resume

In [ ]:
# (Optional) upload checkpoints trained on your local PC, to continue from them here
from google.colab import files
import shutil, os
out = '/content/drive/MyDrive/plump-ai/checkpoints'
os.makedirs(out, exist_ok=True)
for name in files.upload():
    shutil.move(name, os.path.join(out, name))
    print('uploaded', name)

In [ ]:
# (Optional) download the best/latest weights to your local machine
from google.colab import files
import os
base = '/content/drive/MyDrive/plump-ai/checkpoints'
for name in ['best.pt', 'latest.pt']:
    p = os.path.join(base, name)
    if os.path.exists(p):
        files.download(p)
    else:
        print('missing:', p)

## After training (local machine)

Put the downloaded `best.pt`/`latest.pt` into your local `checkpoints/` folder and evaluate or continue:

```bash
.venv/bin/python -m plump.train --iters 20 --players 4 --cards 5 --save-dir checkpoints --resume
```

or just evaluate the saved weights by running a 0-iter resume + the final eval lines it prints.